In [1]:
import pandas as pd
import numpy as np
import time
from datetime import datetime

StatementMeta(, 9c5613fa-bc6f-4d1d-aeb3-3fb2b3a5b3e3, 3, Finished, Available, Finished)

In [2]:
items = pd.read_csv("/lakehouse/default/Files/olist_order_items_dataset.csv")
print(items.head())

StatementMeta(, 9c5613fa-bc6f-4d1d-aeb3-3fb2b3a5b3e3, 4, Finished, Available, Finished)

                           order_id  order_item_id  \
0  00010242fe8c5a6d1ba2dd792cb16214              1   
1  00018f77f2f0320c557190d7a144bdd3              1   
2  000229ec398224ef6ca0657da4fc703e              1   
3  00024acbcdf0a6daa1e931b038114c75              1   
4  00042b26cf59d7ce69dfabb4e55b4fd9              1   

                         product_id                         seller_id  \
0  4244733e06e7ecb4970a6e2683c13e61  48436dade18ac8b2bce089ec2a041202   
1  e5f2d52b802189ee658865ca93d83a8f  dd7ddc04e1b6c2c614352b383efe2d36   
2  c777355d18b72b67abbeef9df44fd0fd  5b51032eddd242adc84c38acab88f23d   
3  7634da152a4610f1595efa32f14722fc  9d7a1d34a5052409006425275ba1c2b4   
4  ac6c3623068f30de03045865e4e10089  df560393f3a51e74553ab94004ba5c87   

   shipping_limit_date   price  freight_value  
0  2017-09-19 09:45:35   58.90          13.29  
1  2017-05-03 11:05:13  239.90          19.93  
2  2018-01-18 14:48:30  199.00          17.87  
3  2018-08-15 10:10:18   12.99          12.7

In [3]:
profile = pd.DataFrame({
    'Column': items.columns.values,
    'negative(%)': [
        len(items[col][items[col] < 0]) / len(items) * 100 if col in items.select_dtypes(include=[np.number]).columns else 0
        for col in items.columns
    ],  
    'zero(%)': [
        len(items[col][items[col] == 0]) / len(items) * 100 if col in items.select_dtypes(include=[np.number]).columns else 0
        for col in items.columns
    ],  
    'duplicates': items.duplicated().sum(), 
    'unique': items.nunique().values, 
})

profile

StatementMeta(, 9c5613fa-bc6f-4d1d-aeb3-3fb2b3a5b3e3, 5, Finished, Available, Finished)

,Column,negative(%),zero(%),duplicates,unique
0,order_id,0.0,0.000000,0,98666
1,order_item_id,0.0,0.000000,0,21
2,product_id,0.0,0.000000,0,32951
3,seller_id,0.0,0.000000,0,3095
4,shipping_limit_date,0.0,0.000000,0,93318
5,price,0.0,0.000000,0,5968
6,freight_value,0.0,0.339991,0,6999


In [4]:
# Insert the data cleaning steps / functions here
# No cleaning necessary.  There are some freight values = 0 but they could be due to some promotion
# Otherwise the table has no null/negative values and no duplicates

items['price']=items['price'].astype(float)
items['freight_value']=items['freight_value'].astype(float)
items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'])

for col in items.columns:
    items[col] = items[col].astype(str)
    items[col] = items[col].str.replace('\xa0', ' ', regex=True)
    items[col] = items[col].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
    items[col] = items[col].str.replace('\r', ' ', regex=True)
    items[col] = items[col].str.replace('\n', ' ', regex=True)
    items[col] = items[col].str.strip()

items_cleaned = items

StatementMeta(, 9c5613fa-bc6f-4d1d-aeb3-3fb2b3a5b3e3, 6, Finished, Available, Finished)

In [6]:
print (items_cleaned.info())

StatementMeta(, 9c5613fa-bc6f-4d1d-aeb3-3fb2b3a5b3e3, 8, Finished, Available, Finished)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   order_id             112650 non-null  object
 1   order_item_id        112650 non-null  object
 2   product_id           112650 non-null  object
 3   seller_id            112650 non-null  object
 4   shipping_limit_date  112650 non-null  object
 5   price                112650 non-null  object
 6   freight_value        112650 non-null  object
dtypes: object(7)
memory usage: 6.0+ MB
None


In [7]:
items_cleaned['shipping_limit_date'] = pd.to_datetime(items_cleaned['shipping_limit_date'])

StatementMeta(, 9c5613fa-bc6f-4d1d-aeb3-3fb2b3a5b3e3, 9, Finished, Available, Finished)

In [9]:
items_cleaned['price'] = items_cleaned['price'].astype(float)
items_cleaned['freight_value'] = items_cleaned['freight_value'].astype(float)

StatementMeta(, 9c5613fa-bc6f-4d1d-aeb3-3fb2b3a5b3e3, 11, Finished, Available, Finished)

In [10]:
from pandas.api.types import is_datetime64_any_dtype

# Check post-cleaned dataframe

assert items_cleaned['order_id'].isnull().sum() == 0, "Null values found in order id"
assert items_cleaned['seller_id'].isnull().sum() == 0, "Null values found in seller id"
assert items_cleaned['price'].isnull().sum() == 0, "Null values found in price"
assert items_cleaned['order_item_id'].isnull().sum() == 0, "Null values found in order item id"
assert is_datetime64_any_dtype(items['shipping_limit_date']), "Column 'shipping_limit_date' is not of datetime type"
assert items_cleaned['price'].dtype == float, "price column is not float"
assert items_cleaned['freight_value'].dtype == float, "freight_value column is not float"

columns_to_check = ['order_id', 'order_item_id', 'seller_id', 'product_id']

# Check if all values in each of these columns are non-zero
assert items_cleaned[columns_to_check].ne(0).all().all(), "Some values are zero in the specified columns"

StatementMeta(, 9c5613fa-bc6f-4d1d-aeb3-3fb2b3a5b3e3, 12, Finished, Available, Finished)

In [12]:
# Write the table to the silver lakehouse as a delta table
# Convert pandas to Spark
spark_items = spark.createDataFrame(items_cleaned)

spark.sql("DROP TABLE IF EXISTS SilverLakehouse.dbo.olist_items_cleaned")

# Save as a Delta table in Silver Lakehouse
silver_path = "SilverLakehouse.dbo.olist_items_cleaned"
spark_items.write.format("delta").mode("overwrite").saveAsTable(silver_path)

StatementMeta(, 9c5613fa-bc6f-4d1d-aeb3-3fb2b3a5b3e3, 14, Finished, Available, Finished)